# Copy/subcopy redundancy curriculum for the LC edit model

The dataset corruption is intentionally route-level: a redundant route is a
copy or a contiguous subcopy of another route.  We do not inject artificial
`last -> y -> last` loops.  The curriculum starts with obvious whole-route
copies, adds boundary subcopies, then mixed interior subcopies and finally
clean LC seeds so the agent also learns when to halt.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # select GPU 1
import sys, pickle, shutil, json, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from hydra import compose, initialize_config_dir
from tqdm.auto import tqdm
from IPython.display import Image, display

from eval_lib.context import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                              MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    _get_planned_current_routes, _make_route_context_state,
    load_raw_graphs_and_lc_routes, make_improvement_batch,
    rollout_lc_improvement, train_lc_improvement_cfg)
from connectpt.routes_generator.torch_utils import (
    get_batch_tensor_from_routes, dump_routes)
from connectpt.routes_generator.transit_time_estimator import (
    ROUTE_ACTION_EXTEND, ROUTE_ACTION_HALT, ROUTE_ACTION_TRIM_END,
    ROUTE_ACTION_TRIM_START, RouteGenBatchState)
from connectpt.routes_generator.citygraph_dataset import (
    STOP_KEY, DynamicCityGraphDataset)
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from torch_geometric.data import Batch
from eval_lib.results_io import save_table
from eval_lib import build_lc_cfg, run_lc, as_route_tensor
from eval_lib import plots as route_plots

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Configuration

`copy_full`, `copy_boundary`, `copy_mixed`, and `lc_clean` are equal-size
tiers.  The three corrupted tiers are generated from LC routes with the four
copy/subcopy mutations below.  Multiplicity grows across the curriculum up to
five routes on one stop-to-stop leg.

In [ ]:
# --- dataset: four equal tiers, easy -> hard/clean ---
# Shared pool: the full run trains on all of it, the route-only probe trains on
# a small per-tier subset (PROBE_N_PER_TIER).
N_GRAPHS        = 500
RAW_N_NODES     = 50
RAW_GRAPH_TYPE  = "mixed"
RAW_GRAPH_SEED  = 0
TARGET_N_ROUTES = 12
MIN_ROUTE_LEN   = 8
MAX_ROUTE_LEN   = 15
LC_N_SAMPLES    = 1
LC_COMBOS = [(1.0,0.0,0.0,"demand"), (0.0,1.0,0.0,"route"), (0.0,0.0,1.0,"conn")]

COPY_MUTATION_KINDS = ("full_copy", "prefix_copy", "suffix_copy", "middle_copy")
TIERS = ["copy_full", "copy_boundary", "copy_mixed", "lc_clean"]
TIER_CFG = {
    "copy_full": {
        "events": 3, "max_multiplicity": 3,
        "kinds": ("full_copy",),
    },
    "copy_boundary": {
        "events": 4, "max_multiplicity": 4,
        "kinds": ("prefix_copy", "suffix_copy"),
    },
    "copy_mixed": {
        "events": 6, "max_multiplicity": 5,
        "kinds": COPY_MUTATION_KINDS,
    },
    "lc_clean": {
        "events": 0, "max_multiplicity": 1,
        "kinds": (),
    },
}
DATASET_DIRNAME = "lc_copy_subcopy_curriculum_n50_r12_len8_15_v2"
NEW_DATASET_DIR = DATASETS_DIR / DATASET_DIRNAME
SUBSET_PKL = NEW_DATASET_DIR / "raw_graphs_subset.pkl"
META_CSV   = NEW_DATASET_DIR / "meta.csv"
FORCE_REGEN = False

# --- objective route + connectivity; Adj stays an eval-only diagnostic ---
DISABLED_COST_COMPONENTS = ["demand"]
ADJ_MODE = "paper"; ADJ_GAP = 0.1
VARY_WEIGHTS = True; OP_FRACTION = 0.4; MCW_FRACTION = 0.4

# --- anti-halt-collapse ---
FORCE_NONHALT_FIRST_STEP = True
FORCE_NONHALT_UNTIL_ITER = 12
# plain differential reward: trim gets its honest per-step cost delta
# (no farm at gamma=1, and trim is not under-credited vs extend).
POSITIVE_ONLY_TRIM_REWARD = False
ZERO_TRIM_REWARD = False
ENTROPY_WEIGHT = 0.01

# --- critic norm + Huber + value clip (used by BOTH runs) ---
CRITIC_OVERRIDES = ["++critic_normalize_returns=true", "++critic_huber=true",
                    "++critic_huber_delta=1.0", "++critic_value_clip=0.2"]

# --- FULL run (route + connectivity, adj off): scaled up ---
N_ITERATIONS = 300
BATCH_SIZE   = 8
TRAIN_FRACTION = 0.9
SPLIT_SEED   = 0
MAX_ROUTE_EDIT_STEPS = MAX_ROUTE_LEN
MAX_TRIM_ACTIONS_PER_ROUTE = 1

# --- resume the full run from the last saved checkpoint (e.g. trained on
# another machine for too few epochs). The continuation's history is written to
# its OWN file(s) so the two parts can be stitched together when plotting. ---
RESUME_FROM_CHECKPOINT = True
RESUME_TAG = "resume"   # suffix for this continuation's history file(s)

# --- PROBE run (route only, no conn / no adj): light sanity check ---
PROBE_N_ITERATIONS = 50
PROBE_N_PER_TIER   = 50    # reduced training subset per tier
PROBE_BATCH_SIZE   = 8

# --- cumulative curriculum: obvious duplicate -> partial overlap -> mixed -> clean ---
# Stage cut-offs scale with N_ITERATIONS (≈16% / 34% / 66% / 100%).
USE_CURRICULUM = True
CURRICULUM = [
    (round(0.16 * N_ITERATIONS), ["copy_full"],                                "full-copy"),
    (round(0.34 * N_ITERATIONS), ["copy_full", "copy_boundary"],               "+boundary"),
    (round(0.66 * N_ITERATIONS), ["copy_full", "copy_boundary", "copy_mixed"], "+mixed"),
    (N_ITERATIONS,               TIERS,                                        "+clean"),
]

# --- evaluation ---
BALANCED_EVAL_WEIGHTS = (0.5, 0.5)
TRAIN_EVAL_N_PER_TIER = 4  # balanced one-batch monitor during training
EVAL_N_PER_TIER = 10
RUN_NAME = "lc_copy_subcopy_curriculum_route_conn_no_adj"
HISTORY_CHECKPOINT_PATH = MODEL_OUTPUTS_DIR / f"{RUN_NAME}_training_history_partial.csv"

# full-run history routing: a fresh run keeps the original file; a resumed run
# writes a tagged continuation file and stitches the original in front at plot.
FULL_HISTORY_RUN = f"{RUN_NAME}_{RESUME_TAG}" if RESUME_FROM_CHECKPOINT else RUN_NAME
FULL_HISTORY_CHECKPOINT = MODEL_OUTPUTS_DIR / f"{FULL_HISTORY_RUN}_training_history_partial.csv"
# prior history CSV(s) to prepend when plotting the full-run curves (machine-1
# part); missing files are skipped gracefully.
PRIOR_HISTORY_FILES = [HISTORY_CHECKPOINT_PATH] if RESUME_FROM_CHECKPOINT else []

print(f"{N_GRAPHS} graphs, tiers={TIERS}; curriculum={[c[2]+'<'+str(c[0]) for c in CURRICULUM]}")
print(f"copy mutations={COPY_MUTATION_KINDS}")
print(f"FULL: route+conn, adj=off, {N_ITERATIONS} epochs, batch={BATCH_SIZE}, "
      f"resume={RESUME_FROM_CHECKPOINT} (history->{FULL_HISTORY_RUN}); "
      f"PROBE: route-only, {PROBE_N_ITERATIONS} epochs, {PROBE_N_PER_TIER}/tier, batch={PROBE_BATCH_SIZE}")
print(f"force_nonhalt<{FORCE_NONHALT_UNTIL_ITER}; "
      f"plain reward (positive_only={POSITIVE_ONLY_TRIM_REWARD}, zero_trim={ZERO_TRIM_REWARD}); "
      f"entropy={ENTROPY_WEIGHT}")
print(f"history checkpoint -> {FULL_HISTORY_CHECKPOINT}")

## Generate the four-tier route-copy dataset

Every corruption copies a whole donor route or a contiguous donor subroute
into another route slot.  Prefix, suffix, and interior replacements preserve
the recipient length.  Candidates with repeated stops are rejected, so the
dataset teaches inter-route redundancy rather than synthetic self-loops.

In [ ]:
def _to_fixed(routes):
    t = as_route_tensor(routes).long()
    if t.ndim == 3:
        t = t[0]
    if t.shape[0] < TARGET_N_ROUTES:
        t = torch.cat([t, torch.full((TARGET_N_ROUTES - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
    else:
        t = t[:TARGET_N_ROUTES]
    if t.shape[1] < MAX_ROUTE_LEN:
        t = torch.cat([t, torch.full((t.shape[0], MAX_ROUTE_LEN - t.shape[1]), -1, dtype=t.dtype)], 1)
    elif t.shape[1] > MAX_ROUTE_LEN:
        t = t[:, :MAX_ROUTE_LEN]
    return t


def _tensors(g):
    return {"node_locs": g[STOP_KEY].pos.detach().cpu().clone(),
            "street_adj": g.street_adj.detach().cpu().clone(),
            "demand": g.demand.detach().cpu().clone()}


def _route_nodes(route):
    return [int(node) for node in route.tolist() if int(node) >= 0]


def _leg_counts(routes):
    counts = Counter()
    for route in routes:
        nodes = _route_nodes(route)
        for start, end in zip(nodes[:-1], nodes[1:]):
            counts[(min(start, end), max(start, end))] += 1
    return counts


def _redundancy_stats(routes):
    counts = _leg_counts(routes)
    traversals = sum(counts.values())
    redundancy = 0.0 if traversals == 0 else (traversals - len(counts)) / traversals
    return {
        "redundancy": float(redundancy),
        "max_leg_use": max(counts.values(), default=0),
        "edge_traversals": traversals,
        "unique_edges": len(counts),
    }


def _is_simple_route(nodes):
    return (MIN_ROUTE_LEN <= len(nodes) <= MAX_ROUTE_LEN and
            len(nodes) == len(set(nodes)) and
            all(a != b for a, b in zip(nodes[:-1], nodes[1:])))


def _copy_candidate(routes, donor_idx, target_idx, kind, rng):
    donor = _route_nodes(routes[donor_idx])
    target = _route_nodes(routes[target_idx])
    if not donor or not target:
        return None
    if kind == "full_copy":
        candidate = donor
    else:
        max_seg_len = min(len(donor), len(target), 6)
        if kind == "middle_copy":
            max_seg_len = min(max_seg_len, len(target) - 2)
        if max_seg_len < 2:
            return None
        seg_len = rng.randint(2, max_seg_len)
        if kind == "prefix_copy":
            candidate = donor[:seg_len] + target[seg_len:]
        elif kind == "suffix_copy":
            candidate = target[:-seg_len] + donor[-seg_len:]
        elif kind == "middle_copy":
            donor_start = rng.randint(0, len(donor) - seg_len)
            target_start = rng.randint(1, len(target) - seg_len - 1)
            candidate = (target[:target_start] +
                         donor[donor_start:donor_start + seg_len] +
                         target[target_start + seg_len:])
        else:
            raise ValueError(f"unknown mutation kind: {kind}")
    if candidate == target or not _is_simple_route(candidate):
        return None
    return candidate


def _replace_route(routes, route_idx, nodes):
    routes[route_idx] = -1
    routes[route_idx, :len(nodes)] = torch.as_tensor(nodes, dtype=routes.dtype)


def _try_copy_mutation(routes, kind, max_multiplicity, rng, attempts=80):
    '''Copy one donor route/subroute into 1..max_multiplicity-1 recipients.'''
    base = routes.clone()
    for _ in range(attempts):
        donor_idx = rng.randrange(routes.shape[0])
        target_idxs = [idx for idx in range(routes.shape[0]) if idx != donor_idx]
        rng.shuffle(target_idxs)
        wanted = rng.randint(1, min(max_multiplicity - 1, len(target_idxs)))
        mutated = base.clone()
        current_redundancy = _redundancy_stats(mutated)["redundancy"]
        touched = 0
        for target_idx in target_idxs:
            candidate = _copy_candidate(mutated, donor_idx, target_idx, kind, rng)
            if candidate is None:
                continue
            proposal = mutated.clone()
            _replace_route(proposal, target_idx, candidate)
            proposal_redundancy = _redundancy_stats(proposal)["redundancy"]
            if proposal_redundancy <= current_redundancy + 1e-12:
                continue
            mutated = proposal
            current_redundancy = proposal_redundancy
            touched += 1
            if touched >= wanted:
                return mutated, touched
    return base, 0


def inject_route_copy_redundancy(routes, tier_cfg, rng):
    routes = routes.clone()
    applied_events = Counter()
    mutated_routes = Counter()
    allowed_kinds = tuple(tier_cfg["kinds"])
    for _ in range(int(tier_cfg["events"])):
        candidates = list(allowed_kinds)
        rng.shuffle(candidates)
        for kind in candidates:
            proposal, touched = _try_copy_mutation(
                routes, kind, int(tier_cfg["max_multiplicity"]), rng)
            if touched:
                routes = proposal
                applied_events[kind] += 1
                mutated_routes[kind] += touched
                break
    return routes, applied_events, mutated_routes


def generate_dataset():
    if NEW_DATASET_DIR.exists():
        shutil.rmtree(NEW_DATASET_DIR)
    NEW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    _random.seed(RAW_GRAPH_SEED); torch.manual_seed(RAW_GRAPH_SEED)
    ds = DynamicCityGraphDataset(min_nodes=RAW_N_NODES, max_nodes=RAW_N_NODES,
                                 data_type=RAW_GRAPH_TYPE, mumford_style=True, pos_only=False)
    raw = [ds.generate_graph(n_nodes=RAW_N_NODES) for _ in range(N_GRAPHS)]
    per = N_GRAPHS // len(TIERS)
    subset, meta = [], []
    for gi, g in enumerate(tqdm(raw, desc="generate dataset")):
        tier = TIERS[min(gi // per, len(TIERS) - 1)]
        tier_cfg = TIER_CFG[tier]
        rng = _random.Random(1000 + gi)
        d, rt, cn, ctag = LC_COMBOS[gi % len(LC_COMBOS)]
        c = build_lc_cfg(run_name=f"copy_cur_{gi}", n_routes=TARGET_N_ROUTES,
                         min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN,
                         demand_time_weight=d, route_time_weight=rt, median_connectivity_weight=cn)
        _, _, _, raw_routes, _ = run_lc(c, tensors=_tensors(g), run_name_prefix="copy_cur_",
                                        n_samples=LC_N_SAMPLES)
        routes = _to_fixed(raw_routes)
        before = _redundancy_stats(routes)
        routes, event_counts, route_counts = inject_route_copy_redundancy(routes, tier_cfg, rng)
        after = _redundancy_stats(routes)
        gdir = NEW_DATASET_DIR / f"graph_{gi:04d}"; gdir.mkdir(parents=True, exist_ok=True)
        dump_routes(f"lc_copy_cur_graph_{gi:04d}_routes", routes, out_dir=gdir)
        subset.append(g)
        meta.append({
            "graph_index": gi, "tier": tier, "lc_combo": ctag,
            "requested_events": int(tier_cfg["events"]),
            "applied_events": int(sum(event_counts.values())),
            "mutated_routes": int(sum(route_counts.values())),
            "mutation_events": json.dumps(dict(event_counts), sort_keys=True),
            "mutation_routes": json.dumps(dict(route_counts), sort_keys=True),
            "redun_before": round(before["redundancy"], 4),
            "redun_after": round(after["redundancy"], 4),
            "max_leg_use_before": before["max_leg_use"],
            "max_leg_use_after": after["max_leg_use"],
        })
        if (gi + 1) % 100 == 0:
            print(f"  {gi+1}/{N_GRAPHS} (tier={tier})")
    with SUBSET_PKL.open("wb") as fh:
        pickle.dump(subset, fh)
    pd.DataFrame(meta).to_csv(META_CSV, index=False)
    print(f"Saved {len(subset)} graphs -> {NEW_DATASET_DIR}")


_have = len(list(NEW_DATASET_DIR.glob("graph_*"))) if NEW_DATASET_DIR.exists() else 0
if SUBSET_PKL.exists() and _have == N_GRAPHS and META_CSV.exists() and not FORCE_REGEN:
    print(f"dataset already exists ({_have}) -> skip")
else:
    if _have and _have != N_GRAPHS:
        print(f"found {_have} graphs, expected {N_GRAPHS} -> regenerate")
    generate_dataset()

## Load, split, and define the cumulative curriculum

In [ ]:
graphs, seed_routes = load_raw_graphs_and_lc_routes(SUBSET_PKL, NEW_DATASET_DIR)
meta_df = pd.read_csv(META_CSV)
N = len(graphs)
print(f"loaded {N} graphs; seed_routes={tuple(seed_routes.shape)}")
print("copy/subcopy corruption summary by tier:")
display(meta_df.groupby("tier")[[
    "requested_events", "applied_events", "mutated_routes",
    "redun_before", "redun_after", "max_leg_use_after",
]].mean().round(3).reindex(TIERS))

_perm = torch.randperm(N, generator=torch.Generator().manual_seed(SPLIT_SEED))
_ntr = int(TRAIN_FRACTION * N)
TRAIN_INDICES = _perm[:_ntr].clone()
VAL_INDICES = _perm[_ntr:].clone()
TIER_OF = dict(zip(meta_df["graph_index"], meta_df["tier"]))

_val_by_tier = {tier: [] for tier in TIERS}
for gi in VAL_INDICES.tolist():
    _val_by_tier[TIER_OF[gi]].append(gi)
if any(len(_val_by_tier[tier]) < TRAIN_EVAL_N_PER_TIER for tier in TIERS):
    raise ValueError("Validation split is too small for the balanced train-loop monitor")
MONITOR_VAL_INDICES = torch.tensor([
    gi for tier in TIERS for gi in _val_by_tier[tier][:TRAIN_EVAL_N_PER_TIER]
], dtype=torch.long)
print(f"validation graphs={len(VAL_INDICES)}; balanced train-loop monitor={len(MONITOR_VAL_INDICES)}")

_train_by_tier = {tier: [] for tier in TIERS}
for gi in TRAIN_INDICES.tolist():
    _train_by_tier[TIER_OF[gi]].append(gi)
_train_by_tier = {
    tier: torch.tensor(indices, dtype=torch.long)
    for tier, indices in _train_by_tier.items()
}
print("train graphs per tier:", {tier: len(indices) for tier, indices in _train_by_tier.items()})


def curriculum_fn(iteration):
    '''iteration -> (active train indices, stage label).'''
    for until, tiers, label in CURRICULUM:
        if iteration < until:
            idx = torch.cat([_train_by_tier[tier] for tier in tiers if len(_train_by_tier[tier])])
            return idx, label
    tiers = CURRICULUM[-1][1]
    idx = torch.cat([_train_by_tier[tier] for tier in tiers if len(_train_by_tier[tier])])
    return idx, CURRICULUM[-1][2]


def stage_spans():
    '''[(start_iter, end_iter, label)] for curriculum shading.'''
    spans, prev = [], 0
    for until, _tiers, label in CURRICULUM:
        spans.append((prev + 1, until, label)); prev = until
    return spans

## Model and objective builders

Two helpers used by both training runs below.  `build_edit_run` composes the
hydra config, builds a **fresh** trim model + cost module and selects which of
the three cost components (`demand` / `route` / `connectivity`) are active.
`train_edit_run` wraps `train_lc_improvement_cfg` and returns the in-memory
history.  Adjustment penalty and conditioning are off in both runs; the trim
model always receives the redundancy-aware edge channels.  The ordinary LC
construction model is untouched.

In [ ]:
def build_edit_run(run_name, disabled_components, vary_weights=True,
                   critic_overrides=None, route_time_weight=None):
    """Compose cfg + build a fresh trim model/cost module for one training run."""
    critic_overrides = list(CRITIC_OVERRIDES if critic_overrides is None
                            else critic_overrides)
    overrides = [
        "model=bestsofar_feb2023_trim",
        "model.route_generator.kwargs.serial_halting=True",
        f"++run_name={run_name}", "++experiment.logdir=null",
        "++adjustment_degree_weight=0.0",
        "++adjustment_conditioning=false",
        "++adjustment_condition_current=false",
        "++adjustment_condition_weight=false",
        f"++entropy_weight={float(ENTROPY_WEIGHT)}",
        f"++force_nonhalt_first_step_until_iter={int(FORCE_NONHALT_UNTIL_ITER)}",
        f"++positive_only_trim_reward={str(POSITIVE_ONLY_TRIM_REWARD).lower()}",
        f"++zero_trim_reward={str(ZERO_TRIM_REWARD).lower()}",
    ] + critic_overrides
    with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
        cfg = compose(config_name="ppo_50nodes.yaml", overrides=overrides)
    _, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
        cfg, run_name_prefix="improvement_")
    cost_obj.ignore_stops_oob = True
    cost_obj.set_enabled_components(disabled_components=disabled_components or None)
    if vary_weights:
        cost_obj.variable_weights = True
        cost_obj.pp_fraction = 0.0
        cost_obj.op_fraction = OP_FRACTION
        cost_obj.mcw_fraction = MCW_FRACTION
    else:
        cost_obj.variable_weights = False
    if route_time_weight is not None:
        cost_obj.route_time_weight = float(route_time_weight)
    best_path = EDIT_MODEL_WEIGHTS_DIR / f"{run_name}.pt"
    print(f"run_name={run_name} | enabled={list(cost_obj.enabled_component_names)} | "
          f"variable_weights={cost_obj.variable_weights} | edge_dim={model.edge_feat_dim}")
    print(f"  trim reward: zero_trim={cfg.get('zero_trim_reward')} "
          f"pos_only={cfg.get('positive_only_trim_reward')} | "
          f"critic_norm={cfg.get('critic_normalize_returns')}")
    return cfg, cost_obj, model, run_name, best_path


def train_edit_run(model, cost_obj, cfg, run_name, best_path,
                   n_iterations, batch_size, train_indices, val_indices,
                   curriculum_fn=None, checkpoint=None):
    """Train one edit model and return its in-memory history DataFrame."""
    result = train_lc_improvement_cfg(
        model=model, cost_obj=cost_obj, graphs=graphs, seed_routes=seed_routes,
        device=device, cfg=cfg, output_dir=MODEL_OUTPUTS_DIR, run_name=run_name,
        train_fraction=TRAIN_FRACTION, batch_size=batch_size,
        min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN, seed=SPLIT_SEED,
        max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
        max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
        target_n_routes=TARGET_N_ROUTES,
        train_indices=train_indices, val_indices=val_indices,
        best_model_path=best_path, n_iterations=n_iterations,
        force_nonhalt_first_step=FORCE_NONHALT_FIRST_STEP,
        curriculum_fn=curriculum_fn,
        history_checkpoint_path=checkpoint,
    )
    df = pd.DataFrame(result["history"])
    save_table(df, f"{run_name}_training_history")
    print(f"history rows={len(df)}; best -> {best_path}")
    return df


print("builders ready: build_edit_run(), train_edit_run()")

## Probe run — route-only (no connectivity, no adj)

A light sanity run that keeps **only the `route` cost component** (demand and
connectivity disabled, adj off). It isolates whether the trim/extend policy can
learn to shorten the network / drop pure route-time duplicates without the
connectivity term protecting overlapping coverage. Simplified settings: a
reduced per-tier subset, no curriculum, fixed `route_time_weight=1.0`, 50
epochs — but the **same critic stack** as the full run (return normalization +
Huber + value clip). Writes its own weights/history so it does not touch the
full-run model below.

In [ ]:
# Route-only probe: only the `route` cost component is active.
probe_cfg, probe_cost, probe_model, probe_run, probe_best = build_edit_run(
    run_name=f"{RUN_NAME}_route_only_probe",
    disabled_components=["demand", "connectivity"],
    vary_weights=False,            # single active component
    route_time_weight=1.0)         # same critic stack (Huber) as the full run

# reduced training subset: PROBE_N_PER_TIER graphs per tier
probe_train_indices = torch.cat([
    _train_by_tier[tier][:PROBE_N_PER_TIER]
    for tier in TIERS if len(_train_by_tier[tier])
])
print(f"probe train graphs={len(probe_train_indices)} "
      f"(<= {PROBE_N_PER_TIER}/tier); epochs={PROBE_N_ITERATIONS}; no curriculum")

probe_history_df = train_edit_run(
    probe_model, probe_cost, probe_cfg, probe_run, probe_best,
    n_iterations=PROBE_N_ITERATIONS, batch_size=PROBE_BATCH_SIZE,
    train_indices=probe_train_indices, val_indices=MONITOR_VAL_INDICES,
    curriculum_fn=None,
    checkpoint=MODEL_OUTPUTS_DIR / f"{probe_run}_training_history_partial.csv")
# probe_model / probe_cost stay alive for the route-only diagnostics below.

## Route-only diagnostics

Everything for the route-only probe: training curves, the per-tier validation
table, and seed→edited route figures (no GIF). Uses `probe_model` / `probe_cost`
and route-only weights (`route=1`, conn/demand off). `ATT` / `CONN` / `d_un` are
still reported as **diagnostics** even though they are not in the route-only
objective — useful to see what shortening route-time does to coverage.

In [ ]:
# Reusable helpers (also handy if you re-run viz without retraining).
def make_weights(cost_obj, demand=0.0, route=1.0, conn=0.0):
    base = cost_obj.get_weights(device)
    w = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in base.items()}
    w["demand_time_weight"] = torch.as_tensor(float(demand), device=device)
    w["route_time_weight"] = torch.as_tensor(float(route), device=device)
    w["median_connectivity_weight"] = torch.as_tensor(float(conn), device=device)
    return w


def _mean_metric(result, key):
    return float(result.get_metrics()[key].detach().float().mean().item())


def evaluate_by_tier(model, cost_obj, weights, n_per_tier, desc="eval"):
    """Per-tier before/after metrics + one visual example per tier."""
    model.eval()
    conn_key = ("median_connectivity_weighted" if cost_obj.use_weighted_connectivity
                else "median_connectivity")
    transfer_keys = {"d0": "$d_0$", "d1": "$d_1$", "d2": "$d_2$", "d_un": "$d_{un}$"}
    cols = ("redun", "ATT", "RTT", "CONN", "d0", "d1", "d2", "d_un")
    rows, examples = [], {}
    for tier in TIERS:
        idxs = _val_by_tier.get(tier, [])[:n_per_tier]
        if len(idxs) == 0:
            continue
        acc = {f"{c}_{when}": [] for c in cols for when in ("before", "after")}
        for gi in tqdm(idxs, desc=f"{desc}/{tier}", leave=False):
            graph_batch, route_batch = make_improvement_batch(
                graphs, seed_routes, torch.tensor([gi]), device,
                training=False, target_n_routes=TARGET_N_ROUTES)
            with torch.no_grad():
                output = rollout_lc_improvement(
                    model, cost_obj, graph_batch, route_batch,
                    MIN_ROUTE_LEN, MAX_ROUTE_LEN, greedy=True, cost_weights=weights,
                    max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
                    max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE)
            final_state, seed_result, final_result = output[:3]
            improved = get_batch_tensor_from_routes(
                final_state.routes, device, max_route_len=route_batch.shape[-1])
            acc["redun_before"].append(_redundancy_stats(route_batch[0])["redundancy"])
            acc["redun_after"].append(_redundancy_stats(improved[0])["redundancy"])
            for name, key in (("ATT", "ATT"), ("RTT", "RTT"), ("CONN", conn_key)):
                acc[f"{name}_before"].append(_mean_metric(seed_result, key))
                acc[f"{name}_after"].append(_mean_metric(final_result, key))
            for name, key in transfer_keys.items():
                acc[f"{name}_before"].append(_mean_metric(seed_result, key))
                acc[f"{name}_after"].append(_mean_metric(final_result, key))
            if tier not in examples:
                nr = min(improved.shape[1], route_batch.shape[1])
                width = min(improved.shape[-1], route_batch.shape[-1])
                adj = get_adjustment_degrees(
                    improved[:, :nr, :width], route_batch[:, :nr, :width],
                    cost_obj.symmetric_routes, gap=ADJ_GAP, mode=ADJ_MODE).mean().item()
                examples[tier] = {
                    "graph_index": gi, "Adj": adj,
                    "seed": route_batch[0].detach().cpu(),
                    "improved": improved[0].detach().cpu(),
                    **{k: acc[k][-1] for k in
                       ("redun_before", "redun_after", "ATT_before", "ATT_after",
                        "RTT_before", "RTT_after", "CONN_before", "CONN_after")},
                }
        rows.append({"tier": tier, "n": len(idxs),
                     **{k: float(np.mean(v)) for k, v in acc.items()}})
    return pd.DataFrame(rows).round(4), examples


def plot_training_curves(h, panels, title, spans=None):
    colors = ["#eaf3ff", "#eafbea", "#fff6e6", "#fdeaea", "#f0eaff"]
    xcol = "epoch" if "epoch" in h.columns else "iteration"
    nrows = (len(panels) + 2) // 3
    fig, ax = plt.subplots(nrows, 3, figsize=(17, 4 * nrows),
                           squeeze=False, constrained_layout=True)
    for a, (col, sub) in zip(ax.flat, panels):
        if spans:
            for k, (s, e, lab) in enumerate(spans):
                a.axvspan(s, e, color=colors[k % len(colors)], alpha=0.6, zorder=0)
        y = pd.to_numeric(h[col], errors="coerce") if col in h.columns else None
        if y is not None and y.notna().any():
            a.plot(h[xcol], y, marker="o", ms=2, color="tab:blue", zorder=3)
        a.axhline(0, color="k", lw=0.7); a.set_title(sub)
        a.set_xlabel(xcol); a.grid(alpha=0.2)
    for a in ax.flat[len(panels):]:
        a.set_visible(False)
    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.show(); plt.close(fig)


def plot_route_examples(examples, title):
    tiers = [t for t in TIERS if t in examples]
    if not tiers:
        print("no examples to plot"); return
    fig, axes = plt.subplots(len(tiers), 2, figsize=(18, 7 * len(tiers)),
                             squeeze=False, constrained_layout=True)
    for r, tier in enumerate(tqdm(tiers, desc="render tiers")):
        ex = examples[tier]; graph = graphs[ex["graph_index"]]
        route_plots.plot_plain_route_set(
            axes[r, 0], ex["seed"], graph,
            title=f"{tier}: corrupted seed (graph {ex['graph_index']})",
            subtitle=(f"redun={ex['redun_before']:.3f}; ATT={ex['ATT_before']:.2f}; "
                      f"RTT={ex['RTT_before']:.2f}; CONN={ex['CONN_before']:.2f}"))
        route_plots.plot_route_diff(
            axes[r, 1], ex["improved"], ex["seed"], graph,
            title=f"{tier}: edited network vs seed",
            subtitle=(f"redun {ex['redun_before']:.3f}->{ex['redun_after']:.3f}; "
                      f"Adj={ex['Adj']:.3f}\n"
                      f"ATT {ex['ATT_before']:.2f}->{ex['ATT_after']:.2f}; "
                      f"RTT {ex['RTT_before']:.2f}->{ex['RTT_after']:.2f}; "
                      f"CONN {ex['CONN_before']:.2f}->{ex['CONN_after']:.2f}"))
    fig.suptitle(title, fontsize=15, fontweight="bold")
    plt.show(); plt.close(fig)


print("route-only diagnostics helpers ready")

In [ ]:
# Route-only training curves (no curriculum -> no stage shading).
# Falls back to the full-run history if the probe was not trained this session.
_diag_hist = probe_history_df if "probe_history_df" in globals() else history_df
_diag_hist_tag = "route-only probe" if "probe_history_df" in globals() else "full run"
route_panels = [
    ("train_reward_mean", "train reward"),
    ("train_return_mean", "train return (per-episode)"),
    ("val_reward", "val reward (per-episode)"),
    ("val_delta", "val cost delta (+=улучш.)"),
    ("val_component_delta_route", "val route-component delta"),
    ("train_action_avg_actions_per_route", "avg edits/route"),
]
plot_training_curves(_diag_hist, route_panels, f"{_diag_hist_tag}: training curves")

# critic curves
_diag_crit = [c for c in _diag_hist.columns if "critic" in c.lower()]
if _diag_crit:
    plot_training_curves(
        _diag_hist, [(c, c) for c in _diag_crit],
        f"{_diag_hist_tag}: critic metrics")

In [ ]:
# Route-only per-tier validation table (route weight = 1; conn/demand = 0).
# Uses the probe model if it is still in memory, otherwise falls back to the
# already-trained full model -> no need to rerun the probe training.
_diag_model = probe_model if "probe_model" in globals() else model
_diag_cost = probe_cost if "probe_cost" in globals() else cost_obj
_diag_tag = "route-only probe" if "probe_model" in globals() else "full model (route-only weights)"
print(f"route-only eval on: {_diag_tag}")

route_only_weights = make_weights(_diag_cost, demand=0.0, route=1.0, conn=0.0)
route_eval_df, route_examples = evaluate_by_tier(
    _diag_model, _diag_cost, route_only_weights, EVAL_N_PER_TIER,
    desc="eval route-only")
display(route_eval_df)
save_table(route_eval_df, f"{globals().get('probe_run', RUN_NAME + '_route_only')}_eval_by_tier")
print("route-only objective = route-time; ATT/RTT/CONN in minutes, "
      "d0/d1/d2/d_un demand% by transfer bucket (CONN/d_un are diagnostics).")

In [ ]:
# Route-only seed -> edited figures, one example per tier.
plot_route_examples(route_examples, "Route-only eval: corruption repair (route objective)")

# Free the probe model ONLY if it exists (no-op when running on the full model).
if "probe_model" in globals():
    del probe_model
if "probe_cost" in globals():
    del probe_cost

## Full run — route + connectivity (adj off), scaled up

The production run: both `route` and `connectivity` components active (demand
off, adj off). Scaled vs the probe — 300 epochs, batch 32, the full 2000-graph
pool, and the complete training stack: cumulative copy→boundary→mixed→clean
curriculum, varying operator/connectivity weight perspectives, critic return
normalization + Huber + value clipping, entropy bonus, and the non-halt
warm-up. This cell sets the `model` / `cost_obj` / `cfg` globals used by the
evaluation and GIF cells below.

In [ ]:
# Full run: route + connectivity active (demand disabled, adj off).
cfg, cost_obj, model, run_name, BEST_MODEL_PATH = build_edit_run(
    run_name=RUN_NAME,
    disabled_components=DISABLED_COST_COMPONENTS,
    vary_weights=VARY_WEIGHTS)

# Resume from the last saved checkpoint (actor weights). The critic re-warms up.
if RESUME_FROM_CHECKPOINT and BEST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    print(f"resumed full-run weights <- {BEST_MODEL_PATH}")
elif RESUME_FROM_CHECKPOINT:
    print(f"RESUME requested but no checkpoint at {BEST_MODEL_PATH}; training from scratch")

history_df = train_edit_run(
    model, cost_obj, cfg, FULL_HISTORY_RUN, BEST_MODEL_PATH,
    n_iterations=N_ITERATIONS, batch_size=BATCH_SIZE,
    train_indices=TRAIN_INDICES, val_indices=MONITOR_VAL_INDICES,
    curriculum_fn=(curriculum_fn if USE_CURRICULUM else None),
    checkpoint=FULL_HISTORY_CHECKPOINT)
print(f"continuation history -> {FULL_HISTORY_RUN}_training_history "
      f"(partial: {FULL_HISTORY_CHECKPOINT.name})")

## Кривые актора (с границами стадий curriculum)

History сохраняется атомарно после каждой завершённой эпохи. При resume этот
прогон пишет свою историю в отдельный файл; для отображения сюда **подшиваются**
прежние части из `PRIOR_HISTORY_FILES` (например, история с первой машины), а ось
эпох делается непрерывной. Если training cell остановлена вручную — можно сразу
выполнить эту ячейку: недостающая in-memory история подхватится из partial CSV.
Заливка стадий curriculum берётся из колонки `curriculum_stage` сшитой истории.

In [ ]:
if "pd" not in globals():
    import pandas as pd
if "plt" not in globals():
    import matplotlib.pyplot as plt
from pathlib import Path

# --- stitch history: prior part(s) (e.g. machine-1) + this run's continuation ---
_parts, _labels = [], []
for _f in (PRIOR_HISTORY_FILES if "PRIOR_HISTORY_FILES" in globals() else []):
    _f = Path(_f)
    if _f.exists():
        _parts.append(pd.read_csv(_f)); _labels.append(f"{_f.name}({len(_parts[-1])})")
if "history_df" in globals():
    _parts.append(history_df.copy()); _labels.append(f"in-memory({len(history_df)})")
elif "FULL_HISTORY_CHECKPOINT" in globals() and Path(FULL_HISTORY_CHECKPOINT).exists():
    _parts.append(pd.read_csv(FULL_HISTORY_CHECKPOINT))
    _labels.append(f"{Path(FULL_HISTORY_CHECKPOINT).name}({len(_parts[-1])})")
if not _parts:
    raise FileNotFoundError("No history found. Train at least one epoch first.")
h = pd.concat(_parts, ignore_index=True)
h["epoch"] = range(1, len(h) + 1)          # continuous axis across stitched parts
print(f"history stitched: {' + '.join(_labels)} => {len(h)} epochs")

def _num(col):
    return pd.to_numeric(h[col], errors="coerce") if col in h.columns else None

# curriculum stage spans derived from the stitched data (robust to stitching)
if "curriculum_stage" in h.columns and h["curriculum_stage"].notna().any():
    _spans = []
    for _epoch, _label in h[["epoch", "curriculum_stage"]].dropna().itertuples(index=False, name=None):
        _epoch = int(_epoch)
        if _spans and _spans[-1][2] == _label:
            _spans[-1] = (_spans[-1][0], _epoch, _label)
        else:
            _spans.append((_epoch, _epoch, _label))
elif "stage_spans" in globals():
    _spans = stage_spans()
else:
    _spans = []
_colors = ["#eaf3ff", "#eafbea", "#fff6e6", "#fdeaea", "#f0eaff"]
def _shade(ax):
    for k, (s, e, lab) in enumerate(_spans):
        ax.axvspan(s, e, color=_colors[k % len(_colors)], alpha=0.6, zorder=0)
        ax.axvline(s, color="gray", lw=0.6, ls=":")

fig, ax = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
panels = [("train_reward_mean","train reward"), ("val_delta","val cost delta (+=улучш.)"),
          ("val_win_rate","val win rate"), ("train_action_avg_actions_per_route","avg edits/route"),
          ("val_component_delta_route","val route delta"),
          ("val_component_delta_connectivity","val conn delta")]
for a,(col,title) in zip(ax.flat, panels):
    _shade(a); y=_num(col)
    if y is not None and y.notna().any():
        a.plot(h["epoch"], y, marker="o", ms=2, color="tab:blue", zorder=3)
    a.axhline(0,color="k",lw=0.7); a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.2)
# подписи стадий сверху
for s,e,lab in _spans:
    ax[0,0].text((s+e)/2, ax[0,0].get_ylim()[1], lab, ha="center", va="bottom", fontsize=8)
fig.suptitle("Actor curves + curriculum stages (заливка = стадия; история сшита)",
             fontsize=13, fontweight="bold")
plt.show(); plt.close(fig)

## Метрики критика (с границами стадий)

In [ ]:
crit_cols = [c for c in h.columns if "critic" in c.lower()]
print("critic columns:", crit_cols)
if crit_cols:
    n=len(crit_cols)
    fig, ax = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False, constrained_layout=True)
    for a, col in zip(ax[0], crit_cols):
        _shade(a); y=_num(col)
        if y is not None and y.notna().any():
            a.plot(h["epoch"], y, marker="o", ms=2, color="tab:orange", zorder=3)
        a.set_title(col, fontsize=9); a.set_xlabel("epoch"); a.grid(alpha=0.2)
        if "explained" in col: a.axhline(0, color="k", lw=0.7)
    fig.suptitle("Critic metrics + curriculum stages", fontsize=13, fontweight="bold")
    plt.show(); plt.close(fig)
    display(h[["epoch","curriculum_stage"]+crit_cols].iloc[::max(1,len(h)//15)].round(4))

## Evaluate balanced policy by tier

The table reports before/after route-level redundancy, `ATT`, `RTT`,
connectivity, and demand percentages by transfer bucket (`d0`, `d1`, `d2`,
`d_un`). `Adj(current, seed)` remains available only in the example plots.

In [ ]:
_required_eval_globals = (
    "TIERS", "EVAL_N_PER_TIER", "_val_by_tier", "BALANCED_EVAL_WEIGHTS",
    "graphs", "seed_routes", "model", "cost_obj", "device",
    "TARGET_N_ROUTES", "MIN_ROUTE_LEN", "MAX_ROUTE_LEN",
    "MAX_ROUTE_EDIT_STEPS", "MAX_TRIM_ACTIONS_PER_ROUTE",
    "_redundancy_stats", "make_improvement_batch", "rollout_lc_improvement",
    "get_batch_tensor_from_routes", "torch", "tqdm",
)
_missing_eval_globals = [name for name in _required_eval_globals if name not in globals()]
if _missing_eval_globals:
    raise RuntimeError(
        "Balanced eval needs initialized config, dataset split, and model. "
        "Run the notebook cells from imports through training first. Missing: "
        + ", ".join(_missing_eval_globals)
    )

def _redun_t(routes_2d):
    return _redundancy_stats(routes_2d)["redundancy"]


def _mean_metric(result, key):
    value = result.get_metrics()[key]
    return float(value.detach().float().mean().item())


val_by_tier = {
    tier: indices[:EVAL_N_PER_TIER]
    for tier, indices in _val_by_tier.items()
}

base_w = cost_obj.get_weights(device)
def mkw(route_weight, conn_weight):
    weights = {key: (value.clone() if torch.is_tensor(value) else value)
               for key, value in base_w.items()}
    weights["demand_time_weight"] = torch.as_tensor(0.0, device=device)
    weights["route_time_weight"] = torch.as_tensor(float(route_weight), device=device)
    weights["median_connectivity_weight"] = torch.as_tensor(float(conn_weight), device=device)
    return weights


rows = []
visual_examples = {}
conn_metric_key = ("median_connectivity_weighted"
                   if cost_obj.use_weighted_connectivity
                   else "median_connectivity")
transfer_metric_keys = {
    "d0": "$d_0$", "d1": "$d_1$", "d2": "$d_2$", "d_un": "$d_{un}$",
}
model.eval()
weights = mkw(*BALANCED_EVAL_WEIGHTS)
for tier in TIERS:
    idxs = val_by_tier[tier]
    if not idxs:
        continue
    metrics = {
        "redun_before": [], "redun_after": [],
        "ATT_before": [], "ATT_after": [],
        "RTT_before": [], "RTT_after": [],
        "CONN_before": [], "CONN_after": [],
        "d0_before": [], "d0_after": [],
        "d1_before": [], "d1_after": [],
        "d2_before": [], "d2_after": [],
        "d_un_before": [], "d_un_after": [],
    }
    for gi in tqdm(idxs, desc=f"eval balanced/{tier}", leave=False):
        graph_batch, route_batch = make_improvement_batch(
            graphs, seed_routes, torch.tensor([gi]), device,
            training=False, target_n_routes=TARGET_N_ROUTES)
        with torch.no_grad():
            output = rollout_lc_improvement(
                model, cost_obj, graph_batch, route_batch,
                MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                greedy=True, cost_weights=weights,
                max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
                max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE)
        final_state, seed_result, final_result = output[:3]
        improved = get_batch_tensor_from_routes(
            final_state.routes, device, max_route_len=route_batch.shape[-1])
        metrics["redun_before"].append(_redun_t(route_batch[0]))
        metrics["redun_after"].append(_redun_t(improved[0]))
        metrics["ATT_before"].append(_mean_metric(seed_result, "ATT"))
        metrics["ATT_after"].append(_mean_metric(final_result, "ATT"))
        metrics["RTT_before"].append(_mean_metric(seed_result, "RTT"))
        metrics["RTT_after"].append(_mean_metric(final_result, "RTT"))
        metrics["CONN_before"].append(_mean_metric(seed_result, conn_metric_key))
        metrics["CONN_after"].append(_mean_metric(final_result, conn_metric_key))
        for metric_name, metric_key in transfer_metric_keys.items():
            metrics[f"{metric_name}_before"].append(_mean_metric(seed_result, metric_key))
            metrics[f"{metric_name}_after"].append(_mean_metric(final_result, metric_key))

        if tier not in visual_examples:
            nr = min(improved.shape[1], route_batch.shape[1])
            width = min(improved.shape[-1], route_batch.shape[-1])
            adj = get_adjustment_degrees(
                improved[:, :nr, :width], route_batch[:, :nr, :width],
                cost_obj.symmetric_routes, gap=ADJ_GAP, mode=ADJ_MODE
            ).mean().item()
            visual_examples[tier] = {
                "graph_index": gi,
                "seed": route_batch[0].detach().cpu(),
                "improved": improved[0].detach().cpu(),
                "Adj": adj,
                "redun_before": metrics["redun_before"][-1],
                "redun_after": metrics["redun_after"][-1],
                "ATT_before": metrics["ATT_before"][-1],
                "ATT_after": metrics["ATT_after"][-1],
                "RTT_before": metrics["RTT_before"][-1],
                "RTT_after": metrics["RTT_after"][-1],
                "CONN_before": metrics["CONN_before"][-1],
                "CONN_after": metrics["CONN_after"][-1],
            }

    means = {key: float(np.mean(values)) for key, values in metrics.items()}
    rows.append({
        "tier": tier, "n": len(idxs),
        "redun_before": means["redun_before"],
        "redun_after": means["redun_after"],
        "ATT_before": means["ATT_before"], "ATT_after": means["ATT_after"],
        "RTT_before": means["RTT_before"], "RTT_after": means["RTT_after"],
        "CONN_before": means["CONN_before"], "CONN_after": means["CONN_after"],
        "d0_before": means["d0_before"], "d0_after": means["d0_after"],
        "d1_before": means["d1_before"], "d1_after": means["d1_after"],
        "d2_before": means["d2_before"], "d2_after": means["d2_after"],
        "d_un_before": means["d_un_before"], "d_un_after": means["d_un_after"],
    })

eval_df = pd.DataFrame(rows).round(4)
display(eval_df)
save_table(eval_df, f"{RUN_NAME}_eval_by_tier")
print("ATT/RTT/CONN are minutes; d0/d1/d2/d_un are demand percentages by transfer bucket.")

## Visual validation examples

For the balanced preference vector, show one seed and the corresponding edited
network from every tier.  The right-hand panels emphasize removed and added
segments relative to the corrupted seed.

In [ ]:
if not visual_examples:
    print("Run the evaluation cell first.")
else:
    tiers_to_plot = [tier for tier in TIERS if tier in visual_examples]
    fig, axes = plt.subplots(
        len(tiers_to_plot), 2,
        figsize=(18, 7 * len(tiers_to_plot)),
        squeeze=False, constrained_layout=True)
    for row_idx, tier in enumerate(tqdm(tiers_to_plot, desc="render tiers")):
        example = visual_examples[tier]
        graph = graphs[example["graph_index"]]
        route_plots.plot_plain_route_set(
            axes[row_idx, 0], example["seed"], graph,
            title=f"{tier}: corrupted seed (graph {example['graph_index']})",
            subtitle=(f"redun={example['redun_before']:.3f}; "
                      f"ATT={example['ATT_before']:.2f}; RTT={example['RTT_before']:.2f}; "
                      f"CONN={example['CONN_before']:.2f}"))
        route_plots.plot_route_diff(
            axes[row_idx, 1], example["improved"], example["seed"], graph,
            title=f"{tier}: edited network vs seed",
            subtitle=(f"redun {example['redun_before']:.3f}->{example['redun_after']:.3f}; "
                      f"Adj={example['Adj']:.3f}\n"
                      f"ATT {example['ATT_before']:.2f}->{example['ATT_after']:.2f}; "
                      f"RTT {example['RTT_before']:.2f}->{example['RTT_after']:.2f}; "
                      f"CONN {example['CONN_before']:.2f}->{example['CONN_after']:.2f}"))
    fig.suptitle("Balanced validation: copy/subcopy corruption repair", fontsize=15, fontweight="bold")
    plt.show()
    plt.close(fig)

## Agent action GIFs

Run the greedy balanced policy step by step for one graph from every tier.
Each GIF starts from the corrupted seed and adds a frame after every agent
action (`extend`, `trim_start`, `trim_end`, or `halt`) with its reward and cost.

In [ ]:
GIF_OUTPUT_DIR = MODEL_OUTPUTS_DIR / "agent_action_gifs"
GIF_FPS = 2
GIF_FORCE_NONHALT_FIRST_STEP = False  # match the balanced eval rollout
ACTION_NAMES = {
    ROUTE_ACTION_EXTEND: "extend", ROUTE_ACTION_TRIM_START: "trim_start",
    ROUTE_ACTION_TRIM_END: "trim_end", ROUTE_ACTION_HALT: "halt",
}


def _is_trim_action(action_kind):
    return int(action_kind) in (ROUTE_ACTION_TRIM_START, ROUTE_ACTION_TRIM_END)


def _action_text(action_kind, action):
    action_kind = int(action_kind)
    name = ACTION_NAMES[action_kind]
    if action_kind == ROUTE_ACTION_HALT:
        return name
    if _is_trim_action(action_kind):
        return f"{name}(position={int(action[0])})"
    return f"{name}({int(action[0])} -> {int(action[1])})"


def _put_route(routes, route_idx, route):
    routes = routes.clone()
    routes[:, route_idx] = -1
    route = route[route > -1].to(routes.device)
    if route.numel():
        route_tensor = get_batch_tensor_from_routes(
            [[route]], routes.device, max_route_len=routes.shape[-1])
        routes[:, route_idx] = route_tensor[:, 0]
    return routes


def _collect_agent_action_frames(graph_index):
    graph_batch, route_batch = make_improvement_batch(
        graphs, seed_routes, torch.tensor([graph_index]), device,
        training=False, target_n_routes=TARGET_N_ROUTES)
    weights = mkw(*BALANCED_EVAL_WEIGHTS)
    reward_scale = float(getattr(cfg, "reward_scale", 1.0))
    diff_reward = bool(getattr(cfg, "diff_reward", True))
    incumbent_reward = bool(getattr(cfg, "incumbent_reward", False))
    zero_trim_reward = bool(getattr(cfg, "zero_trim_reward", False))
    positive_only_trim_reward = bool(getattr(cfg, "positive_only_trim_reward", False))
    edit_step_penalty = float(getattr(cfg, "edit_step_penalty", 0.0))
    forced_halt_penalty = float(getattr(cfg, "forced_halt_penalty", 0.0))
    context_len = max(
        int(route_batch.shape[-1]), int(MAX_ROUTE_LEN),
        int(graph_batch[STOP_KEY].num_nodes))
    working_routes = torch.full(
        (1, route_batch.shape[1], context_len), -1,
        dtype=route_batch.dtype, device=device)
    working_routes[..., :route_batch.shape[-1]] = route_batch
    display_routes = working_routes.clone()
    frames = [{"routes": display_routes[0].detach().cpu(), "label": "corrupted seed"}]
    rows = []

    model.eval()
    with torch.no_grad():
        for route_idx in range(route_batch.shape[1]):
            context_routes = working_routes.clone()
            context_routes[:, route_idx] = -1
            route_state = _make_route_context_state(
                cost_obj, graph_batch, working_routes, route_idx,
                MIN_ROUTE_LEN, MAX_ROUTE_LEN, weights,
                invalid_directly_connected=not bool((context_routes >= 0).any().item()))
            context_counts = route_state.n_finished_routes.detach().clone()
            route_state = model.setup_planning(route_state)
            prev_cost = cost_obj(route_state).cost.detach().clone()
            trim_count = 0

            for step_idx in range(MAX_ROUTE_EDIT_STEPS + 1):
                if route_state.is_done().all():
                    break
                force_halt = step_idx >= MAX_ROUTE_EDIT_STEPS
                trim_allowed = (MAX_TRIM_ACTIONS_PER_ROUTE is None or
                                trim_count < int(MAX_TRIM_ACTIONS_PER_ROUTE))
                if force_halt:
                    step_kinds = torch.full((1,), ROUTE_ACTION_HALT, dtype=torch.long, device=device)
                    step_actions = torch.full((1, 2), -1, dtype=torch.long, device=device)
                else:
                    step_kinds, step_actions, _, _ = model.step_route_action(
                        route_state, greedy=True,
                        allow_halt=not (GIF_FORCE_NONHALT_FIRST_STEP and step_idx == 0),
                        allow_trim_start=trim_allowed, allow_trim_end=trim_allowed)

                cost_before = float(prev_cost.cpu()[0])
                action_kind = int(step_kinds[0].item())
                is_trim = _is_trim_action(action_kind)
                route_state.apply_route_actions(step_kinds, step_actions)
                planned_route = _get_planned_current_routes(
                    route_state, working_routes[:, route_idx], context_counts)[0]
                frame_routes = _put_route(display_routes, route_idx, planned_route)
                new_cost = cost_obj(route_state).cost.detach().clone()
                if incumbent_reward:
                    step_reward = torch.clamp_min(prev_cost - new_cost, 0) * reward_scale
                elif diff_reward:
                    step_reward = (prev_cost - new_cost) * reward_scale
                else:
                    step_reward = torch.zeros_like(new_cost)
                    if action_kind == ROUTE_ACTION_HALT:
                        step_reward = -new_cost * reward_scale
                if positive_only_trim_reward and is_trim:
                    step_reward = step_reward.clamp_min(0)
                elif zero_trim_reward and is_trim:
                    step_reward = torch.zeros_like(step_reward)
                if action_kind != ROUTE_ACTION_HALT:
                    step_reward = step_reward - edit_step_penalty
                if force_halt:
                    step_reward = step_reward - forced_halt_penalty
                if not (zero_trim_reward and not positive_only_trim_reward and is_trim):
                    prev_cost = new_cost
                trim_count += int(is_trim)

                label = (_action_text(action_kind, step_actions[0]) +
                         f" | reward={float(step_reward.cpu()[0]):+.4f}" +
                         f" | cost={float(new_cost.cpu()[0]):.4f}")
                frames.append({"routes": frame_routes[0].detach().cpu(), "label": label})
                rows.append({
                    "route": route_idx, "step": step_idx + 1,
                    "action": _action_text(action_kind, step_actions[0]),
                    "action_kind": ACTION_NAMES[action_kind],
                    "cost_before": cost_before, "cost_after": float(new_cost.cpu()[0]),
                    "reward": float(step_reward.cpu()[0]), "forced_halt": force_halt,
                })
                if action_kind == ROUTE_ACTION_HALT:
                    break

            final_route = _get_planned_current_routes(
                route_state, working_routes[:, route_idx], context_counts)[0]
            display_routes = _put_route(display_routes, route_idx, final_route)
            working_routes = _put_route(working_routes, route_idx, final_route)
    return frames, pd.DataFrame(rows)


def save_agent_action_gif(tier, example):
    frames, steps_df = _collect_agent_action_frames(example["graph_index"])
    graph = graphs[example["graph_index"]]
    fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

    def _draw(frame_idx):
        ax.clear()
        frame = frames[frame_idx]
        route_plots.plot_route_diff(
            ax, frame["routes"], example["seed"], graph,
            title=f"{tier}: greedy balanced actions (graph {example['graph_index']})",
            subtitle=f"frame {frame_idx + 1}/{len(frames)} | {frame['label']}")
        return []

    GIF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    gif_path = GIF_OUTPUT_DIR / f"{RUN_NAME}_{tier}_agent_actions.gif"
    animation = FuncAnimation(
        fig, _draw, frames=len(frames), interval=1000 / GIF_FPS, blit=False)
    try:
        animation.save(gif_path, writer=PillowWriter(fps=GIF_FPS))
        plt.close(fig)
        display(Image(filename=str(gif_path)))
    except Exception as exc:
        plt.close(fig)
        print(f"{tier}: could not save GIF with PillowWriter: {exc}")
        fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
        _draw(len(frames) - 1)
        plt.show(); plt.close(fig)
        gif_path = None
    return gif_path, steps_df


if not visual_examples:
    print("Run the evaluation cell first.")
else:
    saved_gifs, gif_steps_by_tier = {}, {}
    for tier in tqdm([tier for tier in TIERS if tier in visual_examples], desc="render GIFs"):
        gif_path, steps_df = save_agent_action_gif(tier, visual_examples[tier])
        saved_gifs[tier] = gif_path
        gif_steps_by_tier[tier] = steps_df
        save_table(steps_df, f"{RUN_NAME}_{tier}_agent_action_steps")
        print(f"{tier}: {len(steps_df) + 1} frames -> {gif_path}")
        display(steps_df.groupby("action_kind")["reward"].agg(["count", "sum", "mean"]).round(4))